# board_agents

> Task-board channels and numpy agents (Phase 10): `ClaimBoard` (the blackboard for claims), `BoardLead` (the numpy coordinator) and `BoardBandit` (the numpy board agent used for the numpy-vs-LLM comparison).

In [ ]:
#| default_exp board_agents

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from explRL.harness import MessageBus
from explRL.multi import Outcome
from explRL.taskboard import TaskBoardEnv
from explRL.team_agents import PatchMember


class ClaimBoard:
    "Blackboard for claims: the latest 'working' / 'solved' claim per task, with its author and time."
    blocked = frozenset()

    def __init__(self):
        self.claims, self.stats = {}, {}

    def post(self, i, task, kind, t):
        st = self.stats.setdefault(i, dict(written=0, sent=0, received=0))
        if i in self.blocked:
            st['rejected'] = st.get('rejected', 0) + 1
            return
        self.claims[int(task)] = (kind, int(i), float(t))
        st['written'] += 1

    def view(self, i, t, open_):
        "Claims on open tasks by others: [(task, kind, author, age)], newest first."
        out = [(k, c[0], c[1], t - c[2]) for k, c in self.claims.items() if k in open_ and c[1] != i]
        return sorted(out, key=lambda x: (x[3], x[0]))


class BoardLead:
    "Numpy coordinator: suggests the highest-value open task not assigned to another agent (ties to the lower id)."
    def __init__(self, n):
        self.assigned, self.stats = [None] * n, {}

    def assign(self, i, board):
        taken = {x for k, x in enumerate(self.assigned) if k != i}
        free = [k for k in board.open if k not in taken] or list(board.open)
        j = max(free, key=lambda k: (board.tasks[k].value, -k))
        self.assigned[i] = j
        return j


class BoardBandit(PatchMember):
    """Numpy board agent: submits on its open task until it closes; otherwise opens the highest-value open task with
    time left that others have not claimed (or the lead's suggestion), ties at random; posts 'working' / 'solved' claims."""
    extraNames = ('target', 'harvest', 'nHere', 'patch')

    def __init__(self, channel, rng):
        self.channel, self.rng = channel, rng

    def _claimed(self, b, t):
        if isinstance(self.channel, ClaimBoard):
            return {k for k, *_ in self.channel.view(self.id, t, b.open)}
        if isinstance(self.channel, MessageBus):
            self.heard = getattr(self, 'heard', set()) | {j for _, j, _, _ in self.channel.take(self.id)}
            return self.heard & set(b.open)
        return set()

    def choose(self, b, t):
        if b.opened[self.id] in b.open:
            return ('submit', None)
        if isinstance(self.channel, BoardLead):
            return ('open', self.channel.assign(self.id, b))
        live = [k for k in b.open if b.tasks[k].expire > t + 1] or list(b.open)     # time left to submit
        free = [k for k in live if k not in self._claimed(b, t)] or live
        best = max(b.tasks[k].value for k in free)
        top = [k for k in free if b.tasks[k].value == best]
        return ('open', int(top[int(self.rng.integers(len(top)))]))

    def post(self, b, action, info, t):
        kind = 'working' if info.get('result') == 'opened' else 'solved' if info.get('result') == 'solved' else None
        if kind and isinstance(self.channel, ClaimBoard):
            self.channel.post(self.id, info['task'], kind, t)
        elif kind and isinstance(self.channel, MessageBus) and self.channel.due(self.id, t):
            self.channel.send(b, self.id, info['task'], 1.0 if kind == 'working' else 2.0, t)

    def event(self, world, t):
        world.advance(t)
        action = self.choose(world, t)
        reward, dt, loc, info = world.apply(self.id, action, t)
        self.post(world, action, info, t)
        return Outcome(dt, info['kind'], loc, reward, (action[1] if action[1] is not None else -1, info['harvest'],
                                                       info['nHere'], info['patch']))

## Tests

In [ ]:
import numpy as np
from explRL.board_agents import *
from explRL.harness import MessageBus
from explRL.multi import runTeam, spawnRngs
from explRL.taskboard import TaskBoardSim

# one agent with solve rate 1 solves every task it opens
sim = TaskBoardSim(0, 1, solveRate=1.0, difficulty=2)
h = runTeam([BoardBandit(None, np.random.default_rng(0))], sim, 40)
assert len(h) == 40 and sim.counts['solved'] == sim.counts['opens'] == 20 and sim.counts['wrong'] == 0

# claims: the board shows others' latest claims on open tasks, newest first
cb = ClaimBoard()
cb.post(0, 5, 'working', 1.0); cb.post(1, 6, 'working', 2.0); cb.post(1, 5, 'solved', 3.0); cb.post(0, 9, 'working', 3.0)
assert cb.view(0, 4.0, [5, 6, 9]) == [(5, 'solved', 1, 1.0), (6, 'working', 1, 2.0)]
assert cb.view(2, 4.0, [6]) == [(6, 'working', 1, 2.0)] and cb.stats[1]['written'] == 2

# the lead spreads agents over the highest-value tasks
sim = TaskBoardSim(1, 3, solveRate=0.5)
lead = BoardLead(3)
picks = [lead.assign(i, sim) for i in range(3)]
assert len(set(picks)) == 3
vals = sorted((sim.tasks[k].value for k in sim.open), reverse=True)
assert sorted((sim.tasks[k].value for k in picks), reverse=True) == vals[:3]

# channels cut duplicate work: with claims, a team wastes fewer turns on tasks someone else has
def run(channel, seed):
    sim = TaskBoardSim(seed, 4, solveRate=0.3)
    ch = dict(none=None, blackboard=ClaimBoard(), messages=MessageBus(4, 2, 1.0, np.inf, spawnRngs(seed + 2000, 4)),
              hierarchical=BoardLead(4))[channel]
    runTeam([BoardBandit(ch, r) for r in spawnRngs(seed + 1000, 4)], sim, 50)
    return sim.duplicateTurns(), sim.netByAgent().sum()
res = {c: np.mean([run(c, s) for s in range(5)], 0) for c in ('none', 'blackboard', 'messages', 'hierarchical')}
assert res['blackboard'][0] < res['none'][0] and res['hierarchical'][0] < res['none'][0]
assert res['blackboard'][1] > res['none'][1]

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()